# Meta ML-1M SASRec: paper reproduction + realistic latency

Reproduces the frozen Meta `sasrec-sampled-softmax-n128-final` ML-1M recipe before trusting latency. Target: **NDCG@10 ~0.1603**.

Latency is measured on the same trained checkpoint with exact last-200 recomputation and exhaustive raw movie-catalog MIPS + seen-item filtering. The headline is synchronized per-request wall-clock; CUDA Graph is shown only as a lower bound. Checkpoint/resume is stored on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os,sys,shutil,subprocess,torch
REPO='/content/Sparsewalker'; BRANCH='agent/serving-speed-benchmark'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH)


In [ ]:
import runpy
SCRIPT=f'{REPO}/benchmarks/run_meta_sasrec_paper_repro.py'
sys.argv=[SCRIPT,'--epochs','101','--eval-every','10','--batch-size','128']
print('META SASREC REPRO START',flush=True)
runpy.run_path(SCRIPT,run_name='__main__')
print('META SASREC REPRO END',flush=True)


The run is resumable. The important outputs are `META_PROTOCOL`, the final `META_SASREC_QUALITY`, `META_SASREC_LATENCY`, and `META_SASREC_HEADLINE`. If interrupted, rerun the notebook and it resumes from the Drive checkpoint.


In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_meta_sasrec/ml1m/seed42/result.json')
if p.exists():
    r=json.loads(p.read_text())
    print('META_SASREC_HEADLINE',json.dumps(r['headline'],indent=2))
    print('QUALITY',json.dumps(r['quality'],indent=2))
    print('LATENCY',json.dumps(r['latency'],indent=2))
